# Exercise 08a: How much spatial structure does MNIST need?

Tutorial 08a trained one CNN. This exercise uses the same small MNIST sample for a course-level architecture comparison: a flattened MLP, a shallow one-block CNN, and a deeper two-block CNN.

Tune each family independently on validation accuracy. Freeze one final model per family, then evaluate all three on the test set once.


## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `08 Convolutional neural networks`, so relative data paths work.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "08 Convolutional neural networks"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics. PyTorch builds the neural networks used in this lesson.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn torch


In [ ]:
DATA_FILE="mnist_small.npz"

In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42); torch.manual_seed(42); torch.set_num_threads(2)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("device:",DEVICE)

In [ ]:
def train_classifier(model, train_loader, valid_loader, epochs=5, lr=1e-3):
    optimizer=torch.optim.Adam(model.parameters(),lr=lr)
    loss_fn=nn.CrossEntropyLoss(); history={"train_loss":[],"valid_accuracy":[]}
    best_state,best_accuracy=None,-1
    for epoch in range(epochs):
        model.train(); total_loss=0
        for xb,yb in train_loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            optimizer.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); optimizer.step()
            total_loss+=loss.item()*len(xb)
        model.eval(); correct=total=0
        with torch.no_grad():
            for xb,yb in valid_loader:
                pred=model(xb.to(DEVICE)).argmax(1).cpu(); correct+=(pred==yb).sum().item(); total+=len(yb)
        accuracy=correct/total; history["train_loss"].append(total_loss/len(train_loader.dataset)); history["valid_accuracy"].append(accuracy)
        if accuracy>best_accuracy: best_accuracy,best_state=accuracy,copy.deepcopy(model.state_dict())
        print(f"epoch {epoch+1:02d} | training cross-entropy {history['train_loss'][-1]:.4f} | validation accuracy {accuracy:.3f}")
    model.load_state_dict(best_state); return history

def class_predictions(model,X,batch_size=256):
    model.eval(); output=[]
    with torch.no_grad():
        for (xb,) in DataLoader(TensorDataset(torch.tensor(X)),batch_size=batch_size):
            output.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(output)

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self, channels=8, dropout=0.15):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(1,channels,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(channels,2*channels,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(2*channels*7*7,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,10))
    def forward(self,x): return self.classifier(self.features(x))

In [ ]:
data = np.load(DATA_FILE)

# Keep a raw copy for inspection. Pixel values in the source file are integers
# from 0 (black) to 255 (white).
x_train_raw = data["x_train"]
x_valid_raw = data["x_valid"]
x_test_raw = data["x_test"]
y_train = data["y_train"]
y_valid = data["y_valid"]
y_test = data["y_test"]

print("Raw training images:", x_train_raw.shape)
print("Raw pixel range (0=black, 255=white):", x_train_raw.min(), "to", x_train_raw.max())

# Dividing by 255 is useful for neural-network optimisation. It changes only
# the internal pixel scale; class labels and reported accuracy are unchanged.
x_train = (x_train_raw[:, None] / 255).astype("float32")
x_valid = (x_valid_raw[:, None] / 255).astype("float32")
x_test = (x_test_raw[:, None] / 255).astype("float32")

print()
print("Arrays supplied to the CNN:")
print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_valid:", x_valid.shape, "y_valid:", y_valid.shape)
print("x_test :", x_test.shape, "y_test :", y_test.shape)
print("Each x has shape: samples × 1 grayscale channel × 28 rows × 28 columns.")
print("CNN input pixel range after division by 255:", x_train.min(), "to", x_train.max())

## Question 1 — Read the data visually

Plot at least two examples from every class and print class counts. Identify one source of within-class variation and one pair of digits that may be difficult to separate. State the tensor shape expected by a convolutional layer.


In [ ]:
# TODO: display labelled examples and describe the image tensor shape.


## Question 2 — Define three course architectures

Before defining the models, explicitly prepare `X_mlp` with shape $(N,28\times28)=(N,784)$ and `X_cnn` with shape $(N,1,28,28)$. Construct (a) an MLP using `X_mlp`, (b) a shallow CNN using `X_cnn`, and (c) a deeper CNN using `X_cnn`. For a small batch, print each input shape, output shape and parameter count. Explain how local receptive fields, weight sharing, and pooling distinguish the CNNs from the MLP.


In [ ]:
# TODO: define an MLP, shallow CNN and deeper CNN; print input/output shapes and parameters.


## Question 3 — Tune every family on validation data

For the MLP, tune hidden width. For each CNN family, tune channel count while keeping the training budget comparable. Select and freeze one configuration **per family** using validation accuracy. Plot or tabulate validation accuracy and parameter count; do not inspect test labels.


In [ ]:
# TODO: tune one capacity parameter per family using validation accuracy.
# Freeze one setting for each family before looking at test data.


## Question 4 — One final classification comparison

Evaluate all three frozen models on the test set once. Report accuracy and macro F1, then draw a confusion matrix and at least 12 incorrect predictions for the validation-best family. Explain one systematic error pattern without changing the model after seeing test results.


In [ ]:
# TODO: refit/freeze the selected families and evaluate test accuracy once.
# TODO: plot confusion matrices and inspect recurring class confusions.


## Question 5 — Compare learned feature maps

Choose the validation-better of the shallow and deep CNNs. Plot its first-layer filters and activation maps for two images from different classes. Explain what local patterns are visible and why activation magnitude is not a causal explanation.


In [ ]:
# TODO: visualise first-layer feature maps for representative test digits.
# Explain which local patterns the filters respond to.
